In [13]:
import sys
import torch

print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Python: c:\Users\HIMANSHU\Desktop\minor project 1\.venv\Scripts\python.exe
PyTorch: 2.11.0+cu128
CUDA: True
GPU: NVIDIA GeForce RTX 4050 Laptop GPU


In [49]:

# WEEK 3: Deepfake Detection Generalization
# Complete pipeline: data -> split -> balanced training -> evaluation

import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from PIL import Image

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

# --------------------------------------------------
# 1. Setup
# --------------------------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Notebook notebooks/ folder mein hai
ROOT = Path.cwd().parent
CSV_PATH = ROOT / "results" / "face_metadata.csv"

if not CSV_PATH.exists():
    raise FileNotFoundError(
        f"Metadata CSV nahi mili: {CSV_PATH}\n"
        "Notebook ko project ke notebooks folder se run karo."
    )

df = pd.read_csv(CSV_PATH)

required = {"face_path", "video_path", "label"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"CSV mein columns missing hain: {missing}")

print("Metadata loaded:", len(df))
print("\nLabel mapping check:")
if "class_name" in df.columns:
    print(pd.crosstab(df["label"], df["class_name"]))

# Assumption: label 0 = Real, label 1 = Fake
df["label"] = pd.to_numeric(df["label"], errors="coerce")
df = df[df["label"].isin([0, 1])].copy()
df["label"] = df["label"].astype(int)

# --------------------------------------------------
# 2. Find valid image paths
# --------------------------------------------------
def resolve_path(value):
    p = Path(str(value))
    candidates = [
        p,
        ROOT / p,
        ROOT / "results" / p,
        Path.cwd() / p
    ]
    for candidate in candidates:
        if candidate.is_file():
            return str(candidate.resolve())
    return None

df["full_path"] = df["face_path"].apply(resolve_path)
df = df.dropna(subset=["full_path"]).copy()
df = df.reset_index(drop=True)

if len(df) == 0:
    raise ValueError(
        "Koi image path nahi mili. face_path aur project folder check karo."
    )

if df["label"].nunique() != 2:
    raise ValueError("Dataset mein Real aur Fake dono labels hone chahiye.")

print("\nValid images:", len(df))
print("All labels:\n", df["label"].value_counts().sort_index())

# --------------------------------------------------
# 3. Video-wise split: 80% train, 10% validation, 10% test
# --------------------------------------------------
split1 = GroupShuffleSplit(
    n_splits=1, test_size=0.20, random_state=SEED
)
train_idx, temp_idx = next(
    split1.split(df, groups=df["video_path"])
)

train_df = df.iloc[train_idx].copy()
temp_df = df.iloc[temp_idx].copy()

split2 = GroupShuffleSplit(
    n_splits=1, test_size=0.50, random_state=SEED
)
val_idx, test_idx = next(
    split2.split(temp_df, groups=temp_df["video_path"])
)

val_df = temp_df.iloc[val_idx].copy()
test_df = temp_df.iloc[test_idx].copy()

# Confirm no video leakage between splits
train_videos = set(train_df["video_path"])
val_videos = set(val_df["video_path"])
test_videos = set(test_df["video_path"])

assert train_videos.isdisjoint(val_videos)
assert train_videos.isdisjoint(test_videos)
assert val_videos.isdisjoint(test_videos)

print("\nTrain:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

print("\nTrain class counts:")
print(train_df["label"].value_counts().sort_index())
print("\nValidation class counts:")
print(val_df["label"].value_counts().sort_index())
print("\nTest class counts:")
print(test_df["label"].value_counts().sort_index())

# --------------------------------------------------
# 4. Dataset and DataLoaders
# --------------------------------------------------
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor()
])

class FaceDataset(Dataset):
    def __init__(self, data):
        self.data = data.reset_index(drop=True)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        with Image.open(row["full_path"]) as im:
            image = im.convert("RGB")
            image = transform(image)

        label = torch.tensor(float(row["label"]), dtype=torch.float32)
        return image, label

# Balanced sampling: minority class is sampled more often
class_counts = train_df["label"].value_counts()
sample_weights = train_df["label"].map(
    lambda label: 1.0 / class_counts[label]
).to_numpy()

sampler = WeightedRandomSampler(
    weights=torch.as_tensor(sample_weights, dtype=torch.double),
    num_samples=len(train_df),
    replacement=True
)

train_loader = DataLoader(
    FaceDataset(train_df), batch_size=32,
    sampler=sampler, num_workers=0
)
val_loader = DataLoader(
    FaceDataset(val_df), batch_size=32,
    shuffle=False, num_workers=0
)
test_loader = DataLoader(
    FaceDataset(test_df), batch_size=32,
    shuffle=False, num_workers=0
)

print("\nDataLoaders ready")

# --------------------------------------------------
# 5. Custom CNN
# --------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

class CustomCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten()
        )
        self.classifier = nn.Linear(128, 1)

    def forward(self, x):
        return self.classifier(self.features(x)).squeeze(1)

model = CustomCNN().to(device)

# Use balanced sampling instead of adding a second class weight
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

# --------------------------------------------------
# 6. Train and select best validation model
# --------------------------------------------------
EPOCHS = 5
best_val_loss = float("inf")
history = []

models_dir = ROOT / "models"
results_dir = ROOT / "results"
models_dir.mkdir(parents=True, exist_ok=True)
results_dir.mkdir(parents=True, exist_ok=True)

model_path = models_dir / "week3_custom_cnn.pth"

for epoch in range(EPOCHS):
    model.train()
    train_sum = 0.0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        train_sum += loss.item() * labels.size(0)

    model.eval()
    val_sum = 0.0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)
            logits = model(images)
            loss = criterion(logits, labels)
            val_sum += loss.item() * labels.size(0)

    train_loss = train_sum / len(train_loader.dataset)
    val_loss = val_sum / len(val_loader.dataset)

    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "val_loss": val_loss
    })

    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"Train loss: {train_loss:.4f} | "
        f"Val loss: {val_loss:.4f}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), model_path)

pd.DataFrame(history).to_csv(
    results_dir / "week3_training_history.csv", index=False
)

# --------------------------------------------------
# 7. Final test evaluation
# --------------------------------------------------
model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()

y_true, y_pred = [], []
probabilities = []

with torch.no_grad():
    for images, labels in test_loader:
        logits = model(images.to(device))
        probs = torch.sigmoid(logits).cpu().numpy()

        probabilities.extend(probs.tolist())
        y_pred.extend((probs >= 0.5).astype(int).tolist())
        y_true.extend(labels.int().numpy().tolist())

metrics = {
    "accuracy": accuracy_score(y_true, y_pred),
    "precision_fake": precision_score(
        y_true, y_pred, pos_label=1, zero_division=0
    ),
    "recall_fake": recall_score(
        y_true, y_pred, pos_label=1, zero_division=0
    ),
    "f1_fake": f1_score(
        y_true, y_pred, pos_label=1, zero_division=0
    ),
    "recall_real": recall_score(
        y_true, y_pred, pos_label=0, zero_division=0
    )
}

cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

print("\nFinal test metrics:")
for name, value in metrics.items():
    print(f"{name}: {value:.4f}")

print("\nConfusion matrix (rows=true, columns=predicted):")
print("                 Pred Real  Pred Fake")
print(f"Actual Real      {cm[0,0]:8d}  {cm[0,1]:9d}")
print(f"Actual Fake      {cm[1,0]:8d}  {cm[1,1]:9d}")

print("\nClassification report:")
print(classification_report(
    y_true, y_pred,
    labels=[0, 1],
    target_names=["Real", "Fake"],
    zero_division=0
))

pd.DataFrame([metrics]).to_csv(
    results_dir / "week3_custom_cnn_metrics.csv", index=False
)
pd.DataFrame(
    cm, index=["Actual Real", "Actual Fake"],
    columns=["Predicted Real", "Predicted Fake"]
).to_csv(results_dir / "week3_confusion_matrix.csv")

print("\nSaved model:", model_path)
print("Saved results:", results_dir)


Metadata loaded: 61641

Label mapping check:
class_name  deepfake  real
label                     
0                  0  8611
1              53030     0

Valid images: 61641
All labels:
 label
0     8611
1    53030
Name: count, dtype: int64

Train: 49311
Validation: 6160
Test: 6170

Train class counts:
label
0     6691
1    42620
Name: count, dtype: int64

Validation class counts:
label
0     990
1    5170
Name: count, dtype: int64

Test class counts:
label
0     930
1    5240
Name: count, dtype: int64


C:\Users\HIMANSHU\AppData\Local\Temp\ipykernel_18244\353001799.py:165: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  weights=torch.as_tensor(sample_weights, dtype=torch.double),



DataLoaders ready
Device: cuda
Epoch 1/5 | Train loss: 0.6836 | Val loss: 0.6485
Epoch 2/5 | Train loss: 0.6762 | Val loss: 0.6940
Epoch 3/5 | Train loss: 0.6747 | Val loss: 0.6669
Epoch 4/5 | Train loss: 0.6718 | Val loss: 0.6912
Epoch 5/5 | Train loss: 0.6711 | Val loss: 0.6776

Final test metrics:
accuracy: 0.7365
precision_fake: 0.8651
recall_fake: 0.8172
f1_fake: 0.8404
recall_real: 0.2817

Confusion matrix (rows=true, columns=predicted):
                 Pred Real  Pred Fake
Actual Real           262        668
Actual Fake           958       4282

Classification report:
              precision    recall  f1-score   support

        Real       0.21      0.28      0.24       930
        Fake       0.87      0.82      0.84      5240

    accuracy                           0.74      6170
   macro avg       0.54      0.55      0.54      6170
weighted avg       0.77      0.74      0.75      6170


Saved model: c:\Users\HIMANSHU\Desktop\minor project 1\Minor-Project\models\week3_custo

In [50]:
print(pd.crosstab(df["label"], df["class_name"]))

class_name  deepfake  real
label                     
0                  0  8611
1              53030     0
